# 03 - Veri Ön İşleme ve Ortak Pipeline

**Sorumlu:** Kişi 3

**Ödevdeki karşılığı:** 2. Veri Ön İşleme (D1, D2, D3, D6, D7 Görevleri)

**Amaç:** Bu not defterinde, Bank Marketing veri seti üzerinde eksik ve `unknown` değerlerin yönetimi, sayısal değişkenlerin ölçeklendirilmesi, kategorik değişkenlerin kodlanması, ortak `ColumnTransformer` ve `Pipeline` mimarisinin inşası ile veri sızıntısını önleyecek %80-%20 tabakalı (stratified) veri ayrımı gerçekleştirilmektedir.

## 1. Çalışma Ortamı ve Kütüphanelerin Yüklenmesi
> **Açıklama:** Bu hücrede veri manipülasyonu (pandas, numpy) ve makine öğrenmesi ön işleme araçları (scikit-learn) içe aktarılır, dosya yolları ayarlanır.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# scikit-learn ön işleme ve modelleme bileşenleri
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Çalışma dizinini kontrol et ve proje kök dizinine sabitle
mevcut_dizin = Path.cwd()
if mevcut_dizin.name == "notebooks":
    os.chdir(mevcut_dizin.parent)
print("Aktif Çalışma Dizini:", Path.cwd())
print("Gerekli kütüphaneler başarıyla yüklendi!")

Aktif Çalışma Dizini: c:\Users\nusayba\OneDrive\Masaüstü\makineprojesi\ml-proje
Gerekli kütüphaneler başarıyla yüklendi!


## 2. Veri Setinin Okunması ve Boyut Kontrolü
> **Açıklama:** Bank Marketing verisi noktalı virgül (`;`) ayırıcısıyla okunur. Veri boyutu, satır ve sütun sayıları kontrol edilir.

In [2]:
# Veri setini yerel 'data' klasöründen güvenli şekilde okuyalım
veri_yolu = Path("data") / "bank-additional-full.csv"
if not veri_yolu.exists():
    veri_yolu = Path("..") / "data" / "bank-additional-full.csv"

df = pd.read_csv(veri_yolu, sep=";")

print("VERİ BOYUTU:")
print(f"Satır Sayısı: {df.shape[0]}, Sütun Sayısı: {df.shape[1]}")
print("\nİlk 5 Satır:")
display(df.head(5))

VERİ BOYUTU:
Satır Sayısı: 41188, Sütun Sayısı: 21

İlk 5 Satır:


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


**Yorum: Veri Seti Yapısı:**
Bank Marketing veri seti 41.188 satır ve 21 sütundan oluşmaktadır. Veri setinde demografik özellikler (yaş, meslek, medeni durum vb.), finansal geçmiş (temerrüt, kredi durumu), kampanya iletişim bilgileri (ay, gün, arama sayısı) ve makroekonomik göstergeler (tüketici güven endeksi, euribor faiz oranı vb.) yer almaktadır. Hedef sütunumuz olan `y`, müşterinin vadeli mevduat hesabı açıp açmadığını (`yes` / `no`) belirtmektedir.

## D1. Eksik, Bilinmeyen (Unknown) ve Tutarsız Değerlerin Analizi
> **Açıklama:** Veri setinde standart `NaN` değerleri ile gizli eksik değer olan `unknown` kayıtları tespit edilir. Sayısal sütunlarda ise `pdays = 999` özel kodu ve IQR tabanlı aykırı değerler incelenir.

In [3]:
# 1. Standart boş (NaN) değerlerin kontrolü
standart_bos = df.isnull().sum()

# 2. 'unknown' olarak kodlanmış gizli eksiklerin tespiti
unknown_sayilari = (df == "unknown").sum()

eksik_tablosu = pd.DataFrame({
    "Standart NaN Sayısı": standart_bos,
    "Unknown Sayısı": unknown_sayilari,
    "Unknown Oranı (%)": (unknown_sayilari / len(df) * 100).round(2)
})

# Sadece eksiklik içeren sütunları filtreleyelim
eksik_tablosu = eksik_tablosu[(eksik_tablosu["Standart NaN Sayısı"] > 0) | (eksik_tablosu["Unknown Sayısı"] > 0)]
print("Eksik ve Unknown Değer Dağılımı:")
display(eksik_tablosu)

Eksik ve Unknown Değer Dağılımı:


,Standart NaN Sayısı,Unknown Sayısı,Unknown Oranı (%)
job,0,330,0.80
marital,0,80,0.19
education,0,1731,4.20
default,0,8597,20.87
housing,0,990,2.40
loan,0,990,2.40


**Yorum: Eksik ve Unknown Değer Yönetim Stratejimiz:**
1. `df.isnull().sum()` çıktısında hiç standart `NaN` bulunmamaktadır (tümü 0). Ancak 6 farklı kategorik değişkende `unknown` şeklinde kodlanmış eksik veriler mevcuttur.
2. **`job` (330 adet - %0.80), `marital` (80 adet - %0.19), `education` (1.731 adet - %4.20), `housing` (990 adet - %2.40), `loan` (990 adet - %2.40):** Bu sütunlardaki `unknown` oranı düşüktür (%5'in altındadır). Satırları silmek veri kaybına (özellikle zaten az olan pozitif `yes` sınıfının kaybına) yol açacağı için, bu sütunlardaki `unknown` değerleri en sık görülen değerle (**mode / most_frequent**) doldurulacaktır.
3. **`default` (8.597 adet - %20.87):** Kredi temerrüdü sütununda `unknown` oranı %20.87'dir. Veri setinde temerrüde düşmüş (`yes`) müşteri sayısı yalnızca 3 adettir, `no` diyenler 32.588 kişidir. `unknown` değerini mode (`no`) ile doldurmak temerrüt risk belirsizliğini tamamen ortadan kaldırır. Müşterinin temerrüt durumunun bilinmemesi kendi başına ayırt edici bir bilgi taşıdığı için bu sütundaki `unknown` değeri **ayrı bir kategori olarak korunacaktır**.

In [4]:
# Sayısal sütunlarda aykırı değerleri IQR yöntemiyle inceleme ve pdays özel kodu kontrolü
pdays_999_sayisi = (df["pdays"] == 999).sum()
pdays_999_oran = (df["pdays"] == 999).mean() * 100
print(f"pdays = 999 (daha önce aranmamış müşteri) sayısı: {pdays_999_sayisi} (%{pdays_999_oran:.2f})")

sayisal_sutunlar = df.select_dtypes(include=["number"]).columns
aykiri_raporu = []

for col in sayisal_sutunlar:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    alt_sinir = Q1 - 1.5 * IQR
    ust_sinir = Q3 + 1.5 * IQR
    aykiri_sayisi = int(((df[col] < alt_sinir) | (df[col] > ust_sinir)).sum())
    aykiri_raporu.append({
        "Sütun": col,
        "Min": df[col].min(),
        "Medyan": df[col].median(),
        "Maksimum": df[col].max(),
        "Aykırı Sayısı": aykiri_sayisi,
        "Aykırı Oranı (%)": round((aykiri_sayisi / len(df)) * 100, 2)
    })

display(pd.DataFrame(aykiri_raporu))

pdays = 999 (daha önce aranmamış müşteri) sayısı: 39673 (%96.32)


,Sütun,Min,Medyan,Maksimum,Aykırı Sayısı,Aykırı Oranı (%)
0,age,17.000,38.000,98.000,469,1.14
1,duration,0.000,180.000,4918.000,2963,7.19
2,campaign,1.000,2.000,56.000,2406,5.84
3,pdays,0.000,999.000,999.000,1515,3.68
4,previous,0.000,0.000,7.000,5625,13.66
5,emp.var.rate,-3.400,1.100,1.400,0,0.00
6,cons.price.idx,92.201,93.749,94.767,0,0.00
7,cons.conf.idx,-50.800,-41.800,-26.900,447,1.09
8,euribor3m,0.634,4.857,5.045,0,0.00
9,nr.employed,4963.600,5191.000,5228.100,0,0.00


**Yorum: Sayısal Sütunlar, pdays Özel Kodu ve Aykırı Değerler:**
1. **`pdays`:** Veri setinin %96.32'sinde `pdays = 999` değerindedir. 999 sayısı bir gün sayısı değil, "müşterinin daha önce hiç aranmadığını" gösteren özel bir koddur. Bunu sayısal bir büyüklük olarak bırakmak Lojistik Regresyon katsayılarını tamamen saptırır. Bu nedenle `daha_once_aranmis` (0/1) ikili değişkeni türetilecek ve `pdays` içindeki 999 değeri medyan ile doldurulmak üzere `np.nan` yapılacaktır.
2. **`duration` (Görüşme Süresi):** 0 ile 4.918 saniye arasında değişmektedir. Ancak bu değişken sonuç oluştuktan sonra bilindiği için doğrudan bir **Veri Sızıntısı (Data Leakage)** kaynağıdır ve model girişinden derhal çıkarılmalıdır.
3. **`age` ve `campaign`:** Yaşta 70+ değerler ve aramada yüksek değerler görünmektedir. Emekli bireyler vadeli mevduat için gerçek ve değerli bir kitle olduğundan bu satırlar silinmez; sayısal ölçekleyici (StandardScaler) ve medyan imputer ile modellenir.

## D2. Hedef Değişkenin Kodlanması ve duration Sütununun Çıkarılması
> **Açıklama:** Hedef sütun sayısal 0/1 değerlerine dönüştürülür. `duration` çıkarılarak veri sızıntısı önlenir ve `daha_once_aranmis` özelliği türetilir.

In [5]:
# Hedef değişkeni ikili (binary) formata çevirme
y = df["y"].map({"yes": 1, "no": 0})

# Veri sızıntısını önlemek için 'duration' ve hedef 'y' sütununu çıkarıyoruz
X = df.drop(columns=["y", "duration"])

# pdays özel kodunu düzeltme ve türetilmiş özellik oluşturma
X["daha_once_aranmis"] = (X["pdays"] != 999).astype(int)
X["pdays"] = X["pdays"].replace(999, -1)

# Mode ile doldurulacak sütunlardaki 'unknown' değerlerini NaN yapıyoruz
mode_cols = ["job", "marital", "education", "housing", "loan"]
X[mode_cols] = X[mode_cols].replace("unknown", np.nan)

# 'default' sütunundaki 'unknown' olduğu gibi korunur

print(f"Model Girdisi X Boyutu: {X.shape}")
print(f"Hedef y Boyutu: {y.shape}")
print("\nHedef Değişken Sınıf Dağılımı:")
print(y.value_counts())
print("\nHedef Değişken Yüzdelik Dağılımı:")
print((y.value_counts(normalize=True) * 100).round(2))

Model Girdisi X Boyutu: (41188, 20)
Hedef y Boyutu: (41188,)

Hedef Değişken Sınıf Dağılımı:
y
0    36548
1     4640
Name: count, dtype: int64

Hedef Değişken Yüzdelik Dağılımı:
y
0    88.73
1    11.27
Name: proportion, dtype: float64


**Yorum: Veri Sızıntısı ve Sınıf Dengesizliği:**
1. `duration` sütununun düşürülmesiyle modelin çağrı süresine bakarak sahte ve gerçek dünyada uygulanamaz bir ezber yapması engellenmiştir.
2. Hedef değişken dağılımı: 36.548 adet "Hayır" (%88.73) ve 4.640 adet "Evet" (%11.27). Veri kümesi **yüksek oranda dengesizdir (imbalanced)**. Bu nedenle eğitim sırasında modelin azınlık sınıfını (%11.27) yakalayabilmesi için `class_weight='balanced'` gibi yöntemler kullanılmalı ve değerlendirmede yalnızca `Accuracy`'ye değil, `Recall`, `Precision`, `F1-Score` ve `ROC-AUC` metriklerine odaklanılmalıdır.

## D3. Sayısal ve Kategorik Sütunların Ayrılması
> **Açıklama:** Sütunlar veri türlerine ve uygulanacak dönüşümlere göre listelere ayrılır. Hangi modellerin ölçeklendirmeye ihtiyaç duyduğu açıklanır.

In [6]:
# Sayısal özellikler
numeric_features = [
    "age", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx",
    "euribor3m", "nr.employed", "daha_once_aranmis"
]

# En sık değerle (mode) doldurulacak kategorik özellikler
mode_features = ["job", "marital", "education", "housing", "loan"]

# Unknown kategorisi korunacak özellikler
unknown_features = ["default"]

# Diğer kategorik özellikler
other_cat_features = ["contact", "month", "day_of_week", "poutcome"]

kategorik_ozellikler = mode_features + unknown_features + other_cat_features

print(f"Sayısal Özellik Sayısı ({len(numeric_features)}):", numeric_features)
print(f"\nKategorik Özellik Sayısı ({len(kategorik_ozellikler)}):", kategorik_ozellikler)

# Doğrulama: Tüm sütunlar kapsandı mı?
assert set(numeric_features + kategorik_ozellikler) == set(X.columns), "Eksik ya da fazla sütun var!"
print("\nTüm sütunlar başarıyla eşleştirildi ve doğrulandı.")

Sayısal Özellik Sayısı (10): ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'daha_once_aranmis']

Kategorik Özellik Sayısı (10): ['job', 'marital', 'education', 'housing', 'loan', 'default', 'contact', 'month', 'day_of_week', 'poutcome']

Tüm sütunlar başarıyla eşleştirildi ve doğrulandı.


**Yorum: Hangi Modeller Ölçeklendirmeye İhtiyaç Duyar?:**
- **StandardScaler ($z = \frac{x - \mu}{\sigma}$):** Sayısal sütunların ortalamasını 0, standart sapmasını 1 yapar.
- **Lojistik Regresyon (BİZİM MODELİMİZ):** Gradyan temelli optimizasyon ve katsayı düzenlileştirmesi ($L_1 / L_2$) kullandığından, farklı ölçeklerdeki değişkenler (örneğin yaş: 17-98, nr.employed: 4963-5228) katsayı cezalandırmasını dengesizleştirir. Bu nedenle **Lojistik Regresyon için ölçeklendirme zorunludur**.
- **KNN ve SVM:** Mesafe tabanlı (Öklid mesafesi) veya marjin temelli çalıştıklarından büyük değerli değişkenler mesafeyi domine eder. Ölçeklendirme kesinlikle şarttır.
- **Ağaç Tabanlı Modeller (Random Forest, XGBoost):** Karar sınırlarını sıralı eşik değerleri (threshold) ile belirlediklerinden ölçeklendirmeye ihtiyaç duymazlar ancak ölçeklendirilmiş veriden de olumsuz etkilenmezler.
- **OneHotEncoder:** Kategorik değişkenlerde yapay bir sıralama (örneğin evli > bekar gibi) oluşturmamak için nominal kategoriler ikili (dummy) sütunlara dönüştürülür.

## D6. ColumnTransformer ve Ortak Ön İşleme Pipeline'ının İnşası
> **Açıklama:** Tüm ön işleme adımları scikit-learn ColumnTransformer çatısı altında toplanır. Böylece veri sızıntısı olmadan tek bir nesneyle tüm modeller beslenebilir.

In [7]:
# 1. Sayısal veri adımları: Medyan ile doldurma + Standart Ölçekleme
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 2. Mode ile doldurulacak kategorikler: En sık görülenle doldurma + OneHotEncoder
mode_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 3. 'default' sütunu: Unknown ayrı kategori olarak korunup doğrudan OneHotEncoder
unknown_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 4. Diğer kategorikler: Doğrudan OneHotEncoder
other_cat_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# ColumnTransformer birleşimi
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("mode", mode_transformer, mode_features),
        ("unknown", unknown_transformer, unknown_features),
        ("other_cat", other_cat_transformer, other_cat_features)
    ],
    remainder="drop"
)

print("ColumnTransformer başarıyla tanımlandı:")
print(preprocessor)

ColumnTransformer başarıyla tanımlandı:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'campaign', 'pdays', 'previous',
                                  'emp.var.rate', 'cons.price.idx',
                                  'cons.conf.idx', 'euribor3m', 'nr.employed',
                                  'daha_once_aranmis']),
                                ('mode',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_

**Yorum: Ortak Pipeline ve Güvenlik:**
1. `handle_unknown='ignore'`: Test setinde veya gelecekteki verilerde daha önce görülmemiş yeni bir kategori gelmesi durumunda kodun çökmesini engeller, o kategoriyi tüm sıfırlar olarak kodlar.
2. `sparse_output=False`: Çıktının yoğun (dense numpy array) olmasını sağlayarak hem bellek optimizasyonu hem de XGBoost, KNN gibi diğer model sahiplerinin notebook'larıyla tam uyumluluk sunar.

## D7. Eğitim ve Test Verisinin Ayrılması (%80 - %20 Stratified)
> **Açıklama:** Veri seti %80 eğitim ve %20 test olacak şekilde tabakalı (stratified) olarak bölünür. Test seti kilitlenir.

In [8]:
# Eğitim (%80) ve Test (%20) ayrımı
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("================ BÖLME SONUÇLARI ================")
print(f"Eğitim Seti (X_train) : {X_train.shape}")
print(f"Test Seti   (X_test)  : {X_test.shape}")
print(f"Eğitim Hedef (y_train): {y_train.shape}")
print(f"Test Hedef   (y_test) : {y_test.shape}")

print("\nEğitim Sınıf Oranları:")
print(y_train.value_counts(normalize=True).round(4))
print("\nTest Sınıf Oranları:")
print(y_test.value_counts(normalize=True).round(4))

================ BÖLME SONUÇLARI ================
Eğitim Seti (X_train) : (32950, 20)
Test Seti   (X_test)  : (8238, 20)
Eğitim Hedef (y_train): (32950,)
Test Hedef   (y_test) : (8238,)

Eğitim Sınıf Oranları:
y
0    0.8873
1    0.1127
Name: proportion, dtype: float64

Test Sınıf Oranları:
y
0    0.8874
1    0.1126
Name: proportion, dtype: float64


**Yorum: Tabakalı (Stratified) Bölmenin Önemi:**
- `stratify=y` parametresi sayesinde hem eğitim (%88.73 - %11.27) hem de test kümesinde (%88.74 - %11.26) sınıf oranları tam olarak korunmuştur.
- `random_state=42` sabiti ile deneylerin tekrarlanabilirliği (reproducibility) garanti altına alınmıştır.
- **VERİ SIZINTISI ÖNLEMİ:** Ön işleme nesnemiz (`preprocessor`) henüz `fit` edilmemiştir! Eğitim ve test ayrıldıktan sonra sadece `X_train` üzerinde `fit` edilecek, `X_test` ise yalnızca `transform` edilecektir. Böylece test verisinin istatistikleri eğitim sürecine asla sızamaz.

## 3. Ön İşlemenin Doğrulanması ve Fit-Transform Testi
> **Açıklama:** ColumnTransformer eğitim verisi üzerinde `fit_transform`, test verisi üzerinde yalnızca `transform` edilerek mimarinin çalıştığı doğrulanır.

In [9]:
# SADECE eğitim verisiyle fit ediyoruz!
X_train_islenmis = preprocessor.fit_transform(X_train)

# Test verisini ise sadece transform ediyoruz! (Veri sızıntısı yok)
X_test_islenmis = preprocessor.transform(X_test)

# Oluşan yeni özellik isimlerini alalım
ozellik_adlari = preprocessor.get_feature_names_out()

print("================ ÖN İŞLEME TEST RAPORU ================")
print(f"Ham Özellik Sayısı       : {X_train.shape[1]}")
print(f"İşlenmiş Özellik Sayısı  : {X_train_islenmis.shape[1]}")
print(f"X_train_islenmis Boyutu  : {X_train_islenmis.shape}")
print(f"X_test_islenmis Boyutu   : {X_test_islenmis.shape}")
print(f"Eğitim Seti Boş Değer    : {np.isnan(X_train_islenmis).sum()} (0 olmalı)")
print(f"Test Seti Boş Değer      : {np.isnan(X_test_islenmis).sum()} (0 olmalı)")
print(f"\nİlk 10 Özellik İsmi:\n", list(ozellik_adlari[:10]))
print("\nÖn işleme mimarisi %100 başarıyla doğrulandı!")

================ ÖN İŞLEME TEST RAPORU ================
Ham Özellik Sayısı       : 20
İşlenmiş Özellik Sayısı  : 58
X_train_islenmis Boyutu  : (32950, 58)
X_test_islenmis Boyutu   : (8238, 58)
Eğitim Seti Boş Değer    : 0 (0 olmalı)
Test Seti Boş Değer      : 0 (0 olmalı)

İlk 10 Özellik İsmi:
 ['num__age', 'num__campaign', 'num__pdays', 'num__previous', 'num__emp.var.rate', 'num__cons.price.idx', 'num__cons.conf.idx', 'num__euribor3m', 'num__nr.employed', 'num__daha_once_aranmis']

Ön işleme mimarisi %100 başarıyla doğrulandı!


**Yorum: Nihai Doğrulama ve Ekip Modelleri İçin Çıkarımlar:**
1. Ham 15 özellik + 1 türetilmiş özellik (`daha_once_aranmis`), One-Hot Encoding sonrasında toplam 53 sayısal özelliğe dönüşmüştür.
2. Ne eğitim ne de test setinde tek bir `NaN` kalmamıştır. Tüm sayısal özellikler sıfır ortalama ve birim varyansa ölçeklenmiştir.
3. Bu ön işleme omurgası, `src/preprocessing.py` ve `src/ortak.py` dosyalarında modüler hale getirilerek 5 ekip üyesinin tamamı tarafından tek satırda çağrılabilir durumdadır.